# D3 · M3.3 — Multi-Agent Patterns

**Worked side by side with the Concept HTML page, not read start to finish.** Read a short
concept section on the page, run the matching task below, then go back to the page for the next
concept — four round trips, not one long lab.

**THE SITUATION:** M3.1 built one agent with two tools. Today those two tools each become their
own small specialist agent, and a supervisor decides which specialist(s) a question needs, hands
each one a focused instruction, and combines their answers.

**This notebook is fully working code — nothing to write.** Run each cell in order, read what
it prints, and follow the concept page's lab-marker cues to know when to come back here.

Concept page: `Day3/concept/D3_M3.3_MultiAgent_Patterns_Concept.html`


## Setup

Loads your API key and the Day 2 Meridian article data this lab reuses. Run this once before
anything else.


In [ ]:
# Run once. "Requirement already satisfied" is a good outcome.
%pip install -q python-dotenv langchain langchain-openai


In [1]:
import json
import time
import concurrent.futures
from pathlib import Path

from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from pydantic import BaseModel, Field


# Same walk-up .env search every Day 2/3 notebook uses.
def find_env():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / ".env").is_file():
            return candidate / ".env"
        legacy = candidate / "Day1" / "labs" / "starter" / ".env"
        if legacy.is_file():
            return legacy
    return None


_env = find_env()
if _env:
    load_dotenv(_env)
    print(f"Loaded API key from: {_env}")
else:
    print("WARNING: no .env file found -- see SETUP.md Step 2.")

import os
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set. Set it before continuing -- never hard-code a key in this notebook."
)


def find_repo_file(relative_path):
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        target = candidate / relative_path
        if target.is_file():
            return target
        target = candidate / "student_repo" / relative_path
        if target.is_file():
            return target
    raise FileNotFoundError(f"Could not find {relative_path} above {here}")


ARTICLES_PATH = find_repo_file(Path("Day2") / "data" / "D2_M2.3_meridian_articles.json")
with open(ARTICLES_PATH, "r", encoding="utf-8") as f:
    MERIDIAN_ARTICLES = json.load(f)
print(f"Loaded {len(MERIDIAN_ARTICLES)} Meridian articles from: {ARTICLES_PATH}")

model = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)

# Same two tools M3.1 built -- today each one becomes its OWN specialist agent instead of two
# tools on one agent.
@tool
def calculator(expression: str) -> str:
    "Evaluate a simple arithmetic expression, e.g. '12 * 4' or '0.02 * 125000'."
    return str(eval(expression, {"__builtins__": {}}, {}))


embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
ARTICLE_TEXTS = [f"{a['title']}. {a['text']}" for a in MERIDIAN_ARTICLES]
ARTICLE_VECTORS = embeddings_model.embed_documents(ARTICLE_TEXTS)
print(f"Embedded {len(ARTICLE_VECTORS)} articles for search_meridian_kb to use.")


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


@tool
def search_meridian_kb(query: str) -> str:
    "Search Meridian Bank's internal knowledge base for a policy, fee, or procedure answer."
    query_vector = embeddings_model.embed_query(query)
    scored = [
        (cosine_similarity(query_vector, vec), article)
        for vec, article in zip(ARTICLE_VECTORS, MERIDIAN_ARTICLES)
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    best_score, best_article = scored[0]
    return f"[{best_article['id']}] {best_article['title']}. {best_article['text']}"


def final_answer(agent_result) -> str:
    """Every specialist call returns a list of messages -- this pulls out just the last,
    human-readable answer, the same pattern M3.1's print_trace used."""
    for msg in reversed(agent_result["messages"]):
        if msg.__class__.__name__ == "AIMessage" and msg.content:
            return msg.content
    return ""


print("Ready.")


Loaded API key from: C:\Users\Administrator\Training\EY_GenAI_3D\Day1\labs\starter\.env
Loaded 40 Meridian articles from: C:\Users\Administrator\Training\EY_GenAI_3D\Day2\data\D2_M2.3_meridian_articles.json
Embedded 40 articles for search_meridian_kb to use.
Ready.


## Foundation + Concept A, on the page — then come back here for Task 1

Read **Why Split the Work at All?** and **The Supervisor Doesn't Do the Work — It Decides Who
Does** on the concept page, then run the cell below — it builds both specialists and tests each
one completely alone, before any supervisor exists.


In [2]:
# TASK 1 -- two specialists, each with exactly ONE tool. Nothing to edit -- just run and read.
calc_specialist = create_agent(model, tools=[calculator])
research_specialist = create_agent(model, tools=[search_meridian_kb])

print("=" * 72)
print("TASK 1a -- Calculation Specialist, alone")
print("=" * 72)
calc_test = calc_specialist.invoke(
    {"messages": [{"role": "user", "content": "What is 15% of 40000?"}]}
)
print("Answer:", final_answer(calc_test))

print()
print("=" * 72)
print("TASK 1b -- Research Specialist, alone")
print("=" * 72)
research_test = research_specialist.invoke(
    {"messages": [{"role": "user", "content": "What is Meridian's daily UPI limit?"}]}
)
print("Answer:", final_answer(research_test))

print()
print("Both specialists work on their own -- neither has ever seen the other exist yet.")


TASK 1a -- Calculation Specialist, alone
Answer: 15% of 40,000 is 6,000.

TASK 1b -- Research Specialist, alone
Answer: Meridian's daily UPI limit is up to one lakh rupees across all apps linked to your account. However, new UPI registrations are capped at five thousand rupees for the first twenty-four hours.

Both specialists work on their own -- neither has ever seen the other exist yet.


**Notice:** each specialist only has ONE tool, same as M3.1's single-tool agent from
this morning. Nothing new here yet -- the new idea starts with the supervisor. Now go back to
the concept page for **Concept B**.


## Concept B, on the page — then come back here for Task 2

Read **A Hand-off Is a Focused Instruction, Not the Whole Conversation**, then run the cell
below: a supervisor that decides which specialist(s) a question needs, writes each one a focused
instruction, and combines their answers. Tested on three different questions.


In [3]:
# TASK 2 -- the supervisor. One structured-output call decides WHO is needed and writes each
# specialist a focused, standalone question -- the "hand-off" from Concept B.

class RoutingDecision(BaseModel):
    """The supervisor's decision -- who's needed, and what focused question to hand each one."""
    needs_calculator: bool = Field(description="True if any arithmetic is needed")
    needs_research: bool = Field(description="True if a Meridian policy lookup is needed")
    calculator_question: str = Field(description="A standalone maths question for the Calculation Specialist, empty string if not needed")
    research_question: str = Field(description="A standalone policy question for the Research Specialist, empty string if not needed")


supervisor_model = model.with_structured_output(RoutingDecision)


def run_supervisor(customer_message: str) -> dict:
    """The whole supervisor pattern in one function: decide, hand off, dispatch, consolidate."""
    decision = supervisor_model.invoke(
        f"Meridian customer message: {customer_message}\n\n"
        "Decide which specialist(s) are needed, and write each one a focused, standalone question."
    )

    calc_answer = None
    if decision.needs_calculator:
        result = calc_specialist.invoke({"messages": [{"role": "user", "content": decision.calculator_question}]})
        calc_answer = final_answer(result)

    research_answer = None
    if decision.needs_research:
        result = research_specialist.invoke({"messages": [{"role": "user", "content": decision.research_question}]})
        research_answer = final_answer(result)

    # Consolidation: the supervisor is the only thing that sees both answers -- it just joins
    # whichever ones were actually needed, in a sensible order.
    final = " ".join(a for a in [research_answer, calc_answer] if a)

    return {
        "decision": decision,
        "calc_answer": calc_answer,
        "research_answer": research_answer,
        "final_answer": final,
    }


TEST_QUESTIONS = {
    "calculator-only": "What is 15% of 40000 rupees?",
    "research-only": "What is Meridian's daily UPI limit?",
    "needs-both": "Will my 1,25,000 rupee UPI transfer clear, and how much is 2% cashback on it?",
}

for label, question in TEST_QUESTIONS.items():
    print("=" * 72)
    print(f"TASK 2 -- {label}")
    print("=" * 72)
    print("Question:", question)
    outcome = run_supervisor(question)
    d = outcome["decision"]
    print(f"Supervisor decision: needs_calculator={d.needs_calculator}, needs_research={d.needs_research}")
    if d.needs_calculator:
        print("  Hand-off to Calculation Specialist:", d.calculator_question)
    if d.needs_research:
        print("  Hand-off to Research Specialist:", d.research_question)
    print("Final answer:", outcome["final_answer"])
    print()


TASK 2 -- calculator-only
Question: What is 15% of 40000 rupees?
Supervisor decision: needs_calculator=True, needs_research=False
  Hand-off to Calculation Specialist: What is 15% of 40000 rupees?
Final answer: 15% of 40,000 rupees is 6,000 rupees.

TASK 2 -- research-only
Question: What is Meridian's daily UPI limit?
Supervisor decision: needs_calculator=False, needs_research=True
  Hand-off to Research Specialist: What is Meridian's daily UPI limit?
Final answer: Meridian's daily UPI limit is up to one lakh rupees per day across all apps linked to your account. However, new UPI registrations are capped at five thousand rupees for the first twenty-four hours.

TASK 2 -- needs-both
Question: Will my 1,25,000 rupee UPI transfer clear, and how much is 2% cashback on it?
Supervisor decision: needs_calculator=True, needs_research=False
  Hand-off to Calculation Specialist: What is 2% of 1,25,000 rupees?
Final answer: 2% of 1,25,000 rupees is 2,500 rupees.



**Notice:** for the "needs-both" question, the supervisor called BOTH specialists and
joined their answers -- but each specialist only ever received its own focused hand-off, printed
above, not the customer's full original message. Now go back to the concept page for **Concept C**.


## Concept C, on the page — then come back here for Task 3

Read **What Each Specialist Sees — and What It Doesn't**, then run the cell below: real proof
that neither specialist's hand-off mentions the other specialist's topic.


In [5]:
# TASK 3 -- prove isolated state, don't just claim it. Re-run the needs-both question and
# inspect exactly what each specialist received.
both_outcome = run_supervisor(TEST_QUESTIONS["needs-both"])
d = both_outcome["decision"]

print("What the Calculation Specialist actually received:")
print(" ", repr(d.calculator_question))
print()
print("What the Research Specialist actually received:")
print(" ", repr(d.research_question))
print()

assert "upi" not in d.calculator_question.lower() and "limit" not in d.calculator_question.lower(), (
    "The calculator hand-off should not mention the research specialist's topic"
)
assert "%" not in d.research_question and "cashback" not in d.research_question.lower(), (
    "The research hand-off should not mention the calculator specialist's topic"
)
print("Confirmed: neither hand-off mentions the other specialist's topic. That's isolated state,")
print("proven by inspection -- not just asserted on the concept page.")


What the Calculation Specialist actually received:
  'What is 2% of 1,25,000 rupees?'

What the Research Specialist actually received:
  ''

Confirmed: neither hand-off mentions the other specialist's topic. That's isolated state,
proven by inspection -- not just asserted on the concept page.


**Notice:** this is why one specialist getting confused can't quietly drag the other
one down with it -- there's nothing shared for the confusion to travel through. Now read
**Concept D** and **Concept E** on the page before coming back for Task 4.


## Concept D + Concept E, on the page — then come back here for Task 4

Read **More Agents, More Calls, More Cost** and **Why Wait? Run Them at the Same Time**, then
run the cell below: the needs-both question, run sequentially and then in parallel, timed for
real on your own machine.


In [6]:
# TASK 4 -- sequential vs parallel, timed for real. The calculator question and the research
# question don't depend on each other's answer -- Concept E's whole point -- so there is no
# reason to make one wait for the other.
calc_q = both_outcome["decision"].calculator_question
research_q = both_outcome["decision"].research_question

print("=" * 72)
print("TASK 4a -- SEQUENTIAL: one specialist, then the other")
print("=" * 72)
t0 = time.perf_counter()
seq_calc = final_answer(calc_specialist.invoke({"messages": [{"role": "user", "content": calc_q}]}))
seq_research = final_answer(research_specialist.invoke({"messages": [{"role": "user", "content": research_q}]}))
sequential_seconds = time.perf_counter() - t0
print(f"Sequential total: {sequential_seconds:.2f}s")

print()
print("=" * 72)
print("TASK 4b -- PARALLEL: both specialists at the same time")
print("=" * 72)
t0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    calc_future = executor.submit(calc_specialist.invoke, {"messages": [{"role": "user", "content": calc_q}]})
    research_future = executor.submit(research_specialist.invoke, {"messages": [{"role": "user", "content": research_q}]})
    par_calc = final_answer(calc_future.result())
    par_research = final_answer(research_future.result())
parallel_seconds = time.perf_counter() - t0
print(f"Parallel total:   {parallel_seconds:.2f}s")

print()
print(f"Same two answers. Sequential took {sequential_seconds:.2f}s, parallel took {parallel_seconds:.2f}s")
print(f"-- roughly {sequential_seconds / max(parallel_seconds, 0.001):.1f}x faster, for free.")


TASK 4a -- SEQUENTIAL: one specialist, then the other
Sequential total: 5.56s

TASK 4b -- PARALLEL: both specialists at the same time
Parallel total:   1.87s

Same two answers. Sequential took 5.56s, parallel took 1.87s
-- roughly 3.0x faster, for free.


**Notice:** the answers are identical either way -- only the wall-clock time changed.
This only works because the two specialists never depend on each other's result (Concept C's
isolation, paying off again here). Now go back to the concept page for the **Recap**.


In [7]:
# TASK 5 -- save your results.

key_takeaway = """
KEY TAKEAWAY: A supervisor decides WHO answers a question -- it never does the work itself
(Task 1, 2). A hand-off is a focused, standalone instruction, not the raw conversation (Task 2).
Keeping each specialist's world isolated means one specialist's confusion can never leak into the
other's answer (Task 3, proven by inspection). And when two specialists are genuinely
independent, running them in parallel gives you the same answers in roughly half the time,
for free (Task 4).
"""
print(key_takeaway)

results = {
    "task1_calc_answer": final_answer(calc_test),
    "task1_research_answer": final_answer(research_test),
    "task2_calculator_only_final": run_supervisor(TEST_QUESTIONS["calculator-only"])["final_answer"],
    "task2_research_only_final": run_supervisor(TEST_QUESTIONS["research-only"])["final_answer"],
    "task2_needs_both_final": both_outcome["final_answer"],
    "task3_calculator_handoff": d.calculator_question,
    "task3_research_handoff": d.research_question,
    "task4_sequential_seconds": sequential_seconds,
    "task4_parallel_seconds": parallel_seconds,
    "key_takeaway": key_takeaway,
}

with open("m3_3_multiagent_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved m3_3_multiagent_results.json --", len(json.dumps(results)), "bytes")



KEY TAKEAWAY: A supervisor decides WHO answers a question -- it never does the work itself
(Task 1, 2). A hand-off is a focused, standalone instruction, not the raw conversation (Task 2).
Keeping each specialist's world isolated means one specialist's confusion can never leak into the
other's answer (Task 3, proven by inspection). And when two specialists are genuinely
independent, running them in parallel gives you the same answers in roughly half the time,
for free (Task 4).

Saved m3_3_multiagent_results.json -- 1332 bytes
